# 🥉 Bronze Layer Notebook — Incremental Ingestion from Azure SQL

**Layer**: Bronze (Raw Ingestion)  
**Architecture Pattern**: Medallion Architecture  
**Source**: Azure SQL Database via Databricks Lakehouse Federation  
**Target**: Delta Tables in Unity Catalog (`bronze` schema)  
**Load Strategy**: Incremental (watermark-based, not full load)  

---

## 📋 What This Notebook Does

This notebook is the **entry point** of the Medallion pipeline. It reads raw data from an Azure SQL database using Lakehouse Federation and lands it into Delta tables with zero transformation — just as-is from the source, but enriched with audit metadata.

| Step | What Happens |
|------|--------------|
| 1 | Bootstraps schemas and control table on first run |
| 2 | Reads the last successful watermark from the control table |
| 3 | Pulls only new/changed rows from Azure SQL (incremental) |
| 4 | Attaches audit columns: `bronze_ingested_at`, `bronze_run_id`, `source_table` |
| 5 | MERGEs records into Delta tables (`orders_raw`, `products_raw`, `payments_raw`) |
| 6 | Updates the control table with run metadata |

## ⚠️ Key Design Principles
- **Idempotent**: Re-running the same notebook with the same data produces the same result — no duplicates
- **No transformation**: Bronze stores raw data exactly as received from the source
- **Auditable**: Every record knows which run ingested it and when
- **Fault-tolerant**: A failure in one table is recorded and does not block other tables


## Cell 1 — Imports & Spark Configuration

### Why this matters
We import PySpark's `functions` module (aliased `F`) which gives us SQL-style column operations.
`DeltaTable` from the `delta` library is what enables the MERGE/UPSERT pattern — without it,
you'd have to do a full overwrite every time.

We also configure two critical Delta settings:
- `optimizeWrite`: Databricks automatically compacts small files during the write phase (prevents the small-file problem on streaming/incremental workloads)
- `autoCompact`: After each write, Delta runs a background compaction to merge small Parquet files into larger ones (better query performance downstream)
- `shuffle.partitions = 8`: Default is 200 which creates 200 tiny shuffle files on small datasets — tune this to your cluster's core count


In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    LongType, IntegerType
)
from delta.tables import DeltaTable          # Required for MERGE operations
from datetime import datetime, timezone
import uuid
import logging

# ── Logger setup ─────────────────────────────────────────────────────────────
# Using Python logging instead of print() so log level can be controlled
# and messages can be captured by Databricks logging infrastructure
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('BronzeIngestion')

# ── Spark session (already exists in Databricks, no .builder needed) ─────────
spark = SparkSession.builder.getOrCreate()

# ── Delta performance optimisations ──────────────────────────────────────────
# optimizeWrite: Coaleces small files during write (prevents small-file problem)
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')

# autoCompact: Post-write background compaction (better downstream read performance)
spark.conf.set('spark.databricks.delta.autoCompact.enabled', 'true')

# Tune shuffle partitions to cluster size (default 200 is too high for small data)
spark.conf.set('spark.sql.shuffle.partitions', '8')

logger.info('Imports and Spark config complete.')


## Cell 2 — Configuration

### Why centralise config here?
All table names, catalog references, and tuning parameters live in one place. This means:
- **Adding a new source table** = add one dictionary entry to `TABLE_CONFIG`, nothing else changes
- **Switching environments** (dev → staging → prod) = change only `BRONZE_CATALOG` and `FEDERATION_CATALOG`
- No magic strings scattered across 400 lines of code

### Lakehouse Federation — how it works
Databricks Lakehouse Federation creates a **read-only foreign catalog** that maps your Azure SQL tables as if they were native Spark tables. You query them with `spark.table('azuresql_catalog.dbo.orders')` instead of writing JDBC connection strings.
The Federation connector handles the JDBC plumbing, pushes down predicates (WHERE clauses) to SQL Server, and returns a Spark DataFrame.

### Watermark Buffer
The `WATERMARK_BUFFER_MINUTES = 5` setting makes the watermark query look back an extra 5 minutes.
**Why?** Consider this scenario: your last run processed records up to `14:00:00`. Some records with
`updated_at = 13:59:58` were being written to SQL Server exactly as your query ran and didn't appear
in the results. The 5-minute buffer catches those stragglers on the next run, preventing data loss.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Change only this section to adapt to your environment
# ═══════════════════════════════════════════════════════════════════════════

# ── Lakehouse Federation catalog (read-only view into Azure SQL) ─────────────
# This catalog is created via: CREATE FOREIGN CATALOG azuresql_catalog
# USING CONNECTION your_azure_sql_connection;
FEDERATION_CATALOG = 'azuresql_catalog'
FEDERATION_SCHEMA  = 'dbo'

# ── Unity Catalog targets ─────────────────────────────────────────────────────
BRONZE_CATALOG  = 'main'            # Unity Catalog name
BRONZE_SCHEMA   = 'bronze'          # Schema for raw Delta tables
CONTROL_SCHEMA  = 'bronze_control'  # Schema for audit/control tables

# ── Source → Target mapping ───────────────────────────────────────────────────
# primary_key   : Used as the MERGE join key (must be unique in source)
# watermark_col : The timestamp column used to detect new/changed rows
# partition_col : Optional — partitions the Delta table for faster queries
TABLE_CONFIG = {
    'orders': {
        'source_table'  : f'{FEDERATION_CATALOG}.{FEDERATION_SCHEMA}.orders',
        'target_table'  : f'{BRONZE_CATALOG}.{BRONZE_SCHEMA}.orders_raw',
        'primary_key'   : 'order_id',
        'watermark_col' : 'updated_at',
        'partition_col' : 'order_date',   # Partition by date for time-range pruning
    },
    'products': {
        'source_table'  : f'{FEDERATION_CATALOG}.{FEDERATION_SCHEMA}.products',
        'target_table'  : f'{BRONZE_CATALOG}.{BRONZE_SCHEMA}.products_raw',
        'primary_key'   : 'product_id',
        'watermark_col' : 'updated_at',
        'partition_col' : None,           # Products table is small — no partition needed
    },
    'payments': {
        'source_table'  : f'{FEDERATION_CATALOG}.{FEDERATION_SCHEMA}.payments',
        'target_table'  : f'{BRONZE_CATALOG}.{BRONZE_SCHEMA}.payments_raw',
        'primary_key'   : 'payment_id',
        'watermark_col' : 'payment_timestamp',
        'partition_col' : None,
    },
}

# Control table full path
CONTROL_TABLE = f'{BRONZE_CATALOG}.{CONTROL_SCHEMA}.ingestion_control'

# Lookback buffer (minutes) — catches late-arriving records near the watermark boundary
WATERMARK_BUFFER_MINUTES = 5

logger.info('Configuration loaded.')
logger.info(f'Tables to process: {list(TABLE_CONFIG.keys())}')
logger.info(f'Control table: {CONTROL_TABLE}')


## Cell 3 — Schema Bootstrap

### Why `IF NOT EXISTS`?
This makes the notebook **re-runnable from scratch**. If the schemas already exist, Spark skips creation.
If this is the very first run in a new environment, schemas are auto-created.
This is called **idempotent infrastructure** — running it 10 times has the same effect as running it once.

### Why separate `bronze_control` schema?
The control/audit tables have a fundamentally different lifecycle from the raw data tables:
- They are **never cleaned or dropped** even when you wipe and reload bronze
- They serve an operational/monitoring purpose, not a data purpose
- Separating them protects run history from accidental schema-level DROP operations


In [ ]:
def bootstrap_schemas():
    """
    Creates the bronze and control schemas in Unity Catalog if they don't exist.
    Safe to re-run — uses IF NOT EXISTS on all DDL statements.
    """
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}')
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{CONTROL_SCHEMA}')
    logger.info(f'Schema [{BRONZE_CATALOG}.{BRONZE_SCHEMA}] — ready')
    logger.info(f'Schema [{BRONZE_CATALOG}.{CONTROL_SCHEMA}] — ready')

bootstrap_schemas()


## Cell 4 — Control Table Creation

### What is a Control Table and why do we need it?

A **Control Table** is the memory of the pipeline. Without it, every run would have to load the entire
source table from scratch. With it, each run asks: *'What did I last process?'* and reads only
the delta since that point.

### Column-by-column explanation

| Column | Type | Purpose |
|--------|------|---------|
| `table_name` | STRING | Which source table this record tracks |
| `last_processed_timestamp` | TIMESTAMP | Watermark — the max `updated_at` seen in the last successful run. Next run reads `updated_at > this value` |
| `last_processed_pk` | STRING | The max primary key from the last run. Used as a tiebreaker for same-timestamp records |
| `run_id` | STRING | UUID for each execution. Links control records to actual notebook runs |
| `rows_processed` | BIGINT | How many rows were written in this run. Used for anomaly detection (e.g., alert if `rows_processed = 0` unexpectedly) |
| `run_status` | STRING | `STARTED` → `SUCCESS` or `FAILED`. A stuck `STARTED` means the run crashed without cleanup |
| `error_message` | STRING | Full exception message stored on failure — no need to dig through logs |
| `run_start_time` | TIMESTAMP | Pipeline SLA tracking — how long did this run take? |
| `run_end_time` | TIMESTAMP | NULL while running, populated on SUCCESS/FAILED |
| `layer` | STRING | `BRONZE` or `SILVER` — this same pattern is reused for the silver control table |

### Why Delta format for the control table?
Delta gives us ACID transactions. If the notebook crashes mid-write, the control table update
is either fully committed or fully rolled back — never partially written. This means a crashed run
will correctly show as `STARTED` (not `SUCCESS`), which is the right signal for any monitoring system.


In [ ]:
def create_control_table():
    """
    Creates the Bronze ingestion control table if it doesn't exist.
    
    The control table is the pipeline's persistent memory — it stores:
    - WHERE the last run left off (watermark)
    - HOW MANY rows were processed
    - WHETHER the run succeeded or failed
    
    It is stored as a Delta table to benefit from ACID transactions.
    A failed run will NEVER partially update this table.
    """
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
            table_name               STRING    NOT NULL COMMENT 'Source table name being tracked',
            last_processed_timestamp TIMESTAMP          COMMENT 'Max watermark from last successful run',
            last_processed_pk        STRING             COMMENT 'Max primary key from last successful run',
            run_id                   STRING    NOT NULL COMMENT 'UUID for this pipeline execution',
            rows_processed           BIGINT             COMMENT 'Number of rows ingested in this run',
            run_status               STRING    NOT NULL COMMENT 'STARTED | SUCCESS | FAILED',
            error_message            STRING             COMMENT 'Exception detail on failure',
            run_start_time           TIMESTAMP NOT NULL COMMENT 'When this run began',
            run_end_time             TIMESTAMP          COMMENT 'When this run completed (NULL if running)',
            layer                    STRING    NOT NULL COMMENT 'Pipeline layer: BRONZE or SILVER'
        )
        USING DELTA
        COMMENT 'Bronze layer incremental ingestion control and audit table'
        TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'false'
        )
    """)
    logger.info(f'Control table ready: {CONTROL_TABLE}')

create_control_table()


## Cell 5 — Helper Functions

### `get_last_watermark` — The incremental engine
This function is what makes the pipeline incremental. It queries the control table for the last
`SUCCESS` record and returns the watermark timestamp. On the first-ever run, it returns `None`,
which triggers a full load. Every subsequent run uses the stored watermark to read only new rows.

> **Why filter on `run_status = 'SUCCESS'`?**  
> If the last run failed, we don't want to advance the watermark. We want to re-process from the
> last *known good* checkpoint. This automatically gives you **self-healing incremental pipelines** —
> a failed run just means the next run re-tries from the last safe point.

### `write_control_record` — MERGE, not INSERT
We use MERGE instead of INSERT for the control table because:
- If you re-run the same `run_id` (e.g., notebook failed and you re-trigger it), MERGE updates
  the existing record rather than creating a duplicate
- This is what makes the pipeline **idempotent** at the audit layer

### `add_audit_columns` — Bronze provenance tracking
Every row landing in bronze gets three columns:
- `bronze_ingested_at`: Exact timestamp when Databricks wrote this row
- `bronze_run_id`: Links back to the control table record — you can trace any row to its exact run
- `source_table`: Fully qualified source table name — useful when querying across multiple bronze tables

### `read_incremental` — Predicate pushdown to Azure SQL
When you call `spark.table('azuresql_catalog.dbo.orders').filter(col('updated_at') > last_ts)`,
Databricks Federation **pushes the WHERE clause down to SQL Server**. SQL Server executes the filter
and sends only matching rows over the network. You're not pulling the entire table into Spark memory
and filtering there — that would be catastrophically inefficient on large tables.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def get_run_id() -> str:
    """Generate a globally unique run identifier (UUID4)."""
    return str(uuid.uuid4())


def get_last_watermark(table_name: str) -> tuple:
    """
    Retrieves the last successfully committed watermark for a given table.

    Returns:
        (last_timestamp, last_pk) — both None on first run (triggers full load)

    Filters on run_status = 'SUCCESS' deliberately:
    - A FAILED run does not advance the watermark
    - Next run re-processes from the last known-good checkpoint (self-healing)
    """
    result = spark.sql(f"""
        SELECT last_processed_timestamp,
               last_processed_pk
        FROM   {CONTROL_TABLE}
        WHERE  table_name  = '{table_name}'
          AND  run_status  = 'SUCCESS'
          AND  layer       = 'BRONZE'
        ORDER BY run_end_time DESC
        LIMIT 1
    """).collect()

    if result:
        ts = result[0]['last_processed_timestamp']
        pk = result[0]['last_processed_pk']
        logger.info(f'[{table_name}] Last watermark: {ts} | Last PK: {pk}')
        return ts, pk

    logger.info(f'[{table_name}] No prior watermark found — first run.')
    return None, None


def write_control_record(
    table_name: str,
    run_id: str,
    status: str,
    rows_processed: int = 0,
    last_ts=None,
    last_pk: str = None,
    error_msg: str = None,
    run_start=None,
):
    """
    Upserts a single control record for the given run_id + table_name.

    Uses MERGE (not INSERT) so that:
    - Replaying the same run_id is safe — it updates the existing record
    - The control table never accumulates duplicate run records
    - Pipeline re-runs are fully idempotent at the audit layer

    Status flow: STARTED → SUCCESS or FAILED
    """
    now     = datetime.now(timezone.utc)
    run_end = now if status in ('SUCCESS', 'FAILED') else None

    # Build a single-row DataFrame — MERGE requires a DataFrame source
    record_df = spark.createDataFrame([{
        'table_name'               : table_name,
        'last_processed_timestamp' : last_ts,
        'last_processed_pk'        : str(last_pk) if last_pk else None,
        'run_id'                   : run_id,
        'rows_processed'           : rows_processed,
        'run_status'               : status,
        'error_message'            : error_msg,
        'run_start_time'           : run_start or now,
        'run_end_time'             : run_end,
        'layer'                    : 'BRONZE',
    }])

    # MERGE on (run_id + table_name) — composite key prevents cross-table collisions
    DeltaTable.forName(spark, CONTROL_TABLE).alias('ctrl').merge(
        record_df.alias('new'),
        'ctrl.run_id = new.run_id AND ctrl.table_name = new.table_name'
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()


def add_audit_columns(df: DataFrame, run_id: str, source_table: str) -> DataFrame:
    """
    Attaches bronze-layer provenance columns to every ingested row.

    - bronze_ingested_at : Databricks write timestamp (not source timestamp)
    - bronze_run_id      : Links row to control table run record
    - source_table       : Fully qualified source table path

    These columns allow Silver to filter 'what arrived in bronze since my last run'
    using bronze_ingested_at as a reliable pipeline-controlled timestamp.
    """
    return (
        df
        .withColumn('bronze_ingested_at', F.current_timestamp())
        .withColumn('bronze_run_id',      F.lit(run_id))
        .withColumn('source_table',       F.lit(source_table))
    )


def read_incremental(
    source_table: str,
    watermark_col: str,
    last_ts,
    buffer_minutes: int = WATERMARK_BUFFER_MINUTES,
) -> DataFrame:
    """
    Reads only new/changed records from the Federation source.

    How it works:
    1. If last_ts is None (first run): reads the entire table (full load)
    2. If last_ts is set: reads only records where watermark_col > (last_ts - buffer)

    The buffer_minutes subtraction is CRITICAL:
    - It catches records that were being written to SQL Server during the last
      Spark query execution and didn't appear in results (race condition protection)
    - The MERGE step ensures these re-read records don't create duplicates

    Predicate pushdown: The filter is pushed down to SQL Server via Federation,
    so only matching rows are transferred over the network.
    """
    base_df = spark.table(source_table)

    if last_ts is None:
        logger.info(f'[{source_table}] First run — full load.')
        return base_df

    # Subtract buffer from watermark to handle late-arriving records
    adjusted_filter = F.col(watermark_col) > F.expr(
        f"timestampadd(MINUTE, -{buffer_minutes}, CAST('{last_ts}' AS TIMESTAMP))"
    )

    logger.info(
        f'[{source_table}] Incremental read | '
        f'Watermark: {last_ts} | Buffer: -{buffer_minutes}m'
    )
    return base_df.filter(adjusted_filter)


logger.info('Helper functions defined.')


## Cell 6 — MERGE / UPSERT Logic

### Why MERGE and not just append?

Consider what happens without MERGE:
- You run the pipeline at 10:00 AM and ingest order #1234 with status `PENDING`
- The customer pays and order #1234 is updated to `CONFIRMED` in SQL Server
- You run again at 11:00 AM and use `append` mode — now you have **two rows** for order #1234

MERGE solves this: if `order_id` already exists in the Delta table, the row is **updated**.
If it's new, it's **inserted**. The bronze table always has exactly one row per primary key.

### Auto-DDL on first run
We use `spark.catalog.tableExists()` to detect first run. If the table doesn't exist,
we use `saveAsTable()` which infers the schema from the DataFrame — no manual `CREATE TABLE`
DDL required. On subsequent runs, we use `DeltaTable.forName()` for the MERGE path.

### `whenMatchedUpdateAll / whenNotMatchedInsertAll`
These are the two arms of the MERGE:
- `whenMatchedUpdateAll`: When the join condition (`tgt.order_id = src.order_id`) matches,
  overwrite ALL columns on the target with source values
- `whenNotMatchedInsertAll`: When no match, insert the source row as a new record

In a more advanced scenario, you could add `whenMatchedUpdate(condition=..., set=...)` to
only update if `src.updated_at > tgt.updated_at` — preventing older records from overwriting newer ones.


In [ ]:
def upsert_to_bronze(
    source_df: DataFrame,
    target_table: str,
    primary_key: str,
    partition_col: str = None,
) -> int:
    """
    MERGEs a batch of source records into a Bronze Delta table.

    Behaviour:
    - FIRST RUN: Creates the Delta table from the DataFrame schema (auto-DDL)
    - SUBSEQUENT RUNS: Executes a Delta MERGE (upsert) on the primary key

    Returns the number of rows in the source batch (rows_processed for control table).

    Why not just 'append'?
    - Sources update existing records (e.g., order status changes)
    - MERGE ensures each PK has exactly one row — no duplicates
    """
    row_count = source_df.count()

    if row_count == 0:
        logger.info(f'[{target_table}] No new records — skipping merge.')
        return 0

    # ── First run: create the table from DataFrame schema ────────────────────
    if not spark.catalog.tableExists(target_table):
        writer = (
            source_df.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
        )
        # Apply partitioning only if partition column exists in the data
        if partition_col and partition_col in source_df.columns:
            writer = writer.partitionBy(partition_col)

        writer.saveAsTable(target_table)
        logger.info(f'[{target_table}] Created — first run ({row_count:,} rows).')
        return row_count

    # ── Subsequent runs: MERGE on primary key ────────────────────────────────
    # whenMatchedUpdateAll  → updates ALL columns when PK match found (handles source edits)
    # whenNotMatchedInsertAll → inserts the row when PK not found (handles new records)
    delta_target = DeltaTable.forName(spark, target_table)

    (
        delta_target.alias('tgt')
        .merge(
            source_df.alias('src'),
            f'tgt.{primary_key} = src.{primary_key}'  # Join on primary key
        )
        .whenMatchedUpdateAll()      # Source record is newer — overwrite target
        .whenNotMatchedInsertAll()   # Net-new record — insert
        .execute()
    )

    logger.info(f'[{target_table}] MERGE completed — {row_count:,} rows in batch.')
    return row_count


logger.info('Upsert function defined.')


## Cell 7 — Main Processing Function

### The full incremental flow for one table

This function wires together all previous components into a single, self-contained processing unit.
Each table is processed independently so a failure in `payments` doesn't block `orders` or `products`.

```
MARK run as STARTED
    ↓
GET last_watermark from control table
    ↓
READ rows where updated_at > last_watermark (or FULL LOAD if first run)
    ↓
ADD audit columns (bronze_ingested_at, bronze_run_id, source_table)
    ↓
CAPTURE new max watermark from this batch
    ↓
MERGE batch into bronze Delta table
    ↓
MARK run as SUCCESS + store new watermark
    ↓ (on error)
MARK run as FAILED + store error message
```

### Why capture watermark BEFORE the merge?
We compute `max(watermark_col)` from the source batch **before** writing to Delta.
If we computed it after, a concurrent process could have added newer rows to the target table,
and we'd advance the watermark beyond what we actually processed in this run — causing a data gap.

### The `try/except/raise` pattern
On failure, we:
1. Log the error to the control table (persisted to Delta — visible to monitoring)
2. Re-raise the exception so Databricks marks the notebook cell as failed
3. Any downstream job depending on this notebook will also fail-fast

Silently catching exceptions and continuing is an anti-pattern in production pipelines.


In [ ]:
def process_table(table_key: str, cfg: dict, global_run_id: str):
    """
    Orchestrates the full incremental ingestion lifecycle for a single source table.

    This function is idempotent:
    - Re-running with the same global_run_id updates the existing control record
    - Re-running after a failure re-processes from the last successful watermark
    - If the source has no new data, it exits cleanly with rows_processed = 0

    Args:
        table_key      : Key from TABLE_CONFIG ('orders', 'products', 'payments')
        cfg            : Config dict for this table
        global_run_id  : UUID for this pipeline run (shared across all tables)
    """
    table_name = table_key
    # Each table gets a unique run_id derived from the global run
    # This allows per-table tracking while correlating back to the parent run
    run_id    = f'{global_run_id}_{table_key}'
    run_start = datetime.now(timezone.utc)

    logger.info(f"{'='*65}")
    logger.info(f'  Processing: {table_name.upper()}')
    logger.info(f'  Run ID    : {run_id}')
    logger.info(f"{'='*65}")

    # ── Step 1: MARK AS STARTED ───────────────────────────────────────────────
    # Writing STARTED before any work ensures a stuck/crashed run is detectable.
    # A run stuck in STARTED state for >1 hour should trigger an alert.
    write_control_record(
        table_name=table_name, run_id=run_id,
        status='STARTED', run_start=run_start
    )

    try:
        # ── Step 2: GET LAST WATERMARK ────────────────────────────────────────
        # None on first run (triggers full load); timestamp on subsequent runs
        last_ts, last_pk = get_last_watermark(table_name)

        # ── Step 3: INCREMENTAL READ FROM FEDERATION SOURCE ───────────────────
        # This WHERE clause is pushed down to SQL Server via Lakehouse Federation
        # Only rows matching the predicate are transferred over the network
        raw_df = read_incremental(
            source_table  = cfg['source_table'],
            watermark_col = cfg['watermark_col'],
            last_ts       = last_ts,
        )

        # ── Step 4: ATTACH BRONZE AUDIT COLUMNS ───────────────────────────────
        enriched_df = add_audit_columns(raw_df, run_id, cfg['source_table'])

        # ── Step 5: CAPTURE NEW WATERMARK FROM THIS BATCH ─────────────────────
        # Computed BEFORE merge to avoid advancing watermark beyond what we process
        wm_row = enriched_df.agg(
            F.max(cfg['watermark_col']).alias('max_ts'),
            F.max(F.col(cfg['primary_key']).cast('string')).alias('max_pk'),
        ).collect()[0]

        new_max_ts = wm_row['max_ts']
        new_max_pk = wm_row['max_pk']

        # ── Step 6: MERGE INTO BRONZE DELTA TABLE ─────────────────────────────
        rows_processed = upsert_to_bronze(
            source_df    = enriched_df,
            target_table = cfg['target_table'],
            primary_key  = cfg['primary_key'],
            partition_col= cfg.get('partition_col'),
        )

        # ── Step 7: MARK AS SUCCESS + ADVANCE WATERMARK ───────────────────────
        write_control_record(
            table_name    = table_name,
            run_id        = run_id,
            status        = 'SUCCESS',
            rows_processed= rows_processed,
            last_ts       = new_max_ts,
            last_pk       = new_max_pk,
            run_start     = run_start,
        )

        logger.info(f'[{table_name}] ✅  SUCCESS — {rows_processed:,} rows processed.')

    except Exception as e:
        error_msg = str(e)
        logger.error(f'[{table_name}] ❌  FAILED: {error_msg}')

        # Persist error details to control table — visible to monitoring dashboards
        write_control_record(
            table_name = table_name,
            run_id     = run_id,
            status     = 'FAILED',
            error_msg  = error_msg,
            run_start  = run_start,
        )
        raise  # Re-raise so Databricks marks the notebook as failed


logger.info('process_table() function defined.')


## Cell 8 — Pipeline Orchestration & Execution

### Why loop over TABLE_CONFIG instead of writing three separate functions?
- Adding a fourth source table requires zero code changes — just add a config entry
- All tables go through the exact same lifecycle (watermark → read → audit → merge → control)
- Consistent behaviour across tables: if a bug is fixed in `process_table`, all tables benefit

### One `global_run_id` per pipeline execution
All three tables share the same `global_run_id` prefix. This allows you to answer:
*'Show me everything that happened in the 3:00 AM run on 2024-01-15'* by querying
the control table with `WHERE run_id LIKE 'abc123%'`.

### Production deployment note
This notebook is designed to be triggered by:
- A **Databricks Workflow** (Job) on a schedule (e.g., every 30 minutes)
- Or triggered by an **Azure Data Factory** pipeline after the SQL Server transaction commits

When triggered by Databricks Workflows, any unhandled exception here will automatically
mark the Job run as Failed and trigger configured email/Slack/PagerDuty alerts.


In [ ]:
def run_bronze_pipeline():
    """
    Main entry point for the Bronze ingestion pipeline.

    Processes all tables defined in TABLE_CONFIG sequentially.
    Each table is independent — failure in one table does not block others
    (to change this behaviour, remove the individual try/except in process_table
    and let exceptions propagate here).

    A single global_run_id ties all per-table run records together.
    """
    global_run_id = get_run_id()

    logger.info('')
    logger.info('#' * 65)
    logger.info(f'  BRONZE PIPELINE STARTED')
    logger.info(f'  Global Run ID: {global_run_id}')
    logger.info(f'  Tables       : {list(TABLE_CONFIG.keys())}')
    logger.info('#' * 65)

    failed_tables = []

    for table_key, cfg in TABLE_CONFIG.items():
        try:
            process_table(table_key, cfg, global_run_id)
        except Exception as e:
            # Record the failure but continue with remaining tables
            # Remove this try/except if you want pipeline to stop on first failure
            failed_tables.append(table_key)
            logger.error(f'Table [{table_key}] failed: {e}')

    logger.info('')
    logger.info('#' * 65)
    logger.info('  BRONZE PIPELINE COMPLETED')

    if failed_tables:
        logger.error(f'  FAILED TABLES: {failed_tables}')
        raise RuntimeError(f'Bronze pipeline failed for tables: {failed_tables}')
    else:
        logger.info('  ALL TABLES: SUCCESS ✅')
    logger.info('#' * 65)


# ── EXECUTE ────────────────────────────────────────────────────────────────
run_bronze_pipeline()


## Cell 9 — Post-Run Validation

### Why validate after writing?
A successful `run_bronze_pipeline()` means the code ran without exceptions — but it doesn't guarantee
the data actually landed correctly. These validation checks catch issues like:
- A table that should have 50,000 rows but only has 3 (Federation connectivity issue)
- A run that completed with `rows_processed = 0` when new data was expected
- A table that exists but hasn't been updated in the expected time window

In production, these checks would feed into a monitoring dashboard or trigger alerts.
For now, `display()` makes results visible in the Databricks notebook UI.


In [ ]:
# ── Row count summary across all bronze tables ──────────────────────────────
print('\n📊 Bronze Table Row Counts:')
print('-' * 50)
for table_key, cfg in TABLE_CONFIG.items():
    try:
        cnt = spark.table(cfg['target_table']).count()
        print(f'  {cfg["target_table"]:<50} {cnt:>10,} rows')
    except Exception as e:
        print(f'  {cfg["target_table"]:<50} ERROR: {e}')

# ── Control table summary — last run status for each table ───────────────────
print('\n📋 Latest Control Table Records:')
display(spark.sql(f"""
    SELECT
        table_name,
        run_status,
        rows_processed,
        last_processed_timestamp AS watermark,
        ROUND((unix_timestamp(run_end_time) - unix_timestamp(run_start_time)), 1) AS duration_sec,
        run_end_time
    FROM   {CONTROL_TABLE}
    WHERE  layer = 'BRONZE'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY table_name ORDER BY run_end_time DESC) = 1
    ORDER BY table_name
"""))
